# Fixed vs. Dynamic Few-Shot — Comparison

Compares **fixed** few-shot prompting (hand-written examples, from the static
runs) against **dynamic** few-shot prompting (per-question retrieved examples,
from the dynamic runs), across all models, on the 100-question
`facebook/natural_reasoning` test set.

Reads only the results files under `Results/` — no dataset, no Drive:

| Technique | Fixed (static) | Dynamic |
|---|---|---|
| Few-shot | `Results/static/results_<model>.json` → `few_shot` | `Results/dynamic/results_<model>_dynamic.json` → `dynamic_few_shot` |
| Few-shot CoT | `... → few_shot_cot` | `... → dynamic_few_shot_cot` |

**Metrics:** Exact Match, Contains Match, ROUGE-L, BERTScore F1, F1 token overlap —
all recomputed here from the stored responses with one shared code path, so fixed
and dynamic are scored identically.

> `no_answer` questions (18) are excluded from scoring (no reference answer).
> Models missing either their static or dynamic results file are skipped, so this
> runs as soon as any model has both. DeepSeek slots in automatically once its
> runs land.

## 1. Setup

In [ ]:
print("Dependencies handled by the .venv")

In [ ]:
import os

os.environ.setdefault("HF_HOME", os.path.abspath(".hf_cache"))

import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from rouge_score import rouge_scorer
from bert_score import score as compute_bert_score

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

## 2. Configuration & data loading

In [ ]:
PROJECT_PATH = "."
ANSWER_MARKER = "ANSWER:"

MODELS = [
    ("llama",    "Llama 3.1 8B"),
    ("mistral",  "Mistral 7B v0.3"),
    ("gemma",    "Gemma 4 E4B"),
    ("deepseek", "DeepSeek 7B Chat"),
]
MODEL_ORDER = [label for _, label in MODELS]

# The two few-shot techniques, each in a fixed and a dynamic variant. The map
# turns the four raw technique names (across both results files) into a
# (base, source) pair.
BASE_ORDER = ["few_shot", "few_shot_cot"]
BASE_LABELS = {"few_shot": "Few-shot", "few_shot_cot": "Few-shot CoT"}
SOURCE_ORDER = ["fixed", "dynamic"]
TECH_MAP = {
    "few_shot":             ("few_shot", "fixed"),
    "few_shot_cot":         ("few_shot_cot", "fixed"),
    "dynamic_few_shot":     ("few_shot", "dynamic"),
    "dynamic_few_shot_cot": ("few_shot_cot", "dynamic"),
}

METRICS = ["exact_match", "contains_match", "rouge_l", "bertscore_f1", "f1_token"]
METRIC_LABELS = {
    "exact_match": "Exact Match",
    "contains_match": "Contains Match",
    "rouge_l": "ROUGE-L",
    "bertscore_f1": "BERTScore F1",
    "f1_token": "F1 Token",
}
ANSWER_TYPE_ORDER = ["single_word", "short", "long"]

In [ ]:
frames, loaded, skipped = [], [], []
for stem, label in MODELS:
    static_path  = f"{PROJECT_PATH}/Results/static/results_{stem}.json"
    dynamic_path = f"{PROJECT_PATH}/Results/dynamic/results_{stem}_dynamic.json"
    if not (os.path.exists(static_path) and os.path.exists(dynamic_path)):
        skipped.append(label)
        continue
    parts = []
    for path in (static_path, dynamic_path):
        with open(path, encoding="utf-8") as f:
            parts.append(pd.DataFrame(json.load(f)))
    frame = pd.concat(parts, ignore_index=True)
    frame = frame[frame["technique"].isin(TECH_MAP)].copy()
    frame["base"]   = frame["technique"].map(lambda t: TECH_MAP[t][0])
    frame["source"] = frame["technique"].map(lambda t: TECH_MAP[t][1])
    frame["model"]  = label
    frames.append(frame)
    loaded.append(label)

assert frames, "No model has both a static and a dynamic results file yet."
present = [l for l in MODEL_ORDER if l in loaded]

df = pd.concat(frames, ignore_index=True)
df["model"]  = pd.Categorical(df["model"], categories=present, ordered=True)
df["base"]   = pd.Categorical(df["base"], categories=BASE_ORDER, ordered=True)
df["source"] = pd.Categorical(df["source"], categories=SOURCE_ORDER, ordered=True)
MODEL_ORDER = present

print(f"Loaded: {loaded}")
if skipped:
    print(f"Skipped (missing static or dynamic results): {skipped}")
print(f"\n{len(df)} rows total.")
print(df.groupby(["model", "source"], observed=True)["base"].value_counts().unstack())

## 3. Evaluation functions

Copied verbatim from the per-model notebooks so the recomputed scores match what each run reported.

In [ ]:
def normalize_latex(text):
    text = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', text)
    text = re.sub(r'\\boxed\{([^}]+)\}', r'\1', text)
    text = re.sub(r'\$+', '', text)
    text = re.sub(r'\\(left|right|quad|qquad|text|mathrm|mathbf|displaystyle)\b', '', text)
    text = re.sub(r'\\{2,}', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_answer(response):
    if ANSWER_MARKER in response:
        return response.split(ANSWER_MARKER)[-1].strip()

    boxed = re.findall(r'\\boxed\{([^}]+)\}', response)
    if boxed:
        return boxed[-1]

    patterns = [
        r'(?:the\s+)?(?:final\s+)?answer\s+is[:\s]+(.+?)(?:\.|$)',
        r'therefore[,:\s]+(.+?)(?:\.|$)',
    ]
    for pattern in patterns:
        match = re.search(pattern, response, re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip()

    sentences = [s.strip() for s in response.rstrip('.').split('.') if s.strip()]
    return sentences[-1] if sentences else response

def normalize(text):
    text = normalize_latex(text)
    return text.lower().strip().rstrip('.')

In [ ]:
rouge_l_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def exact_match(prediction, reference):
    return normalize(extract_answer(prediction)) == normalize(reference)

def contains_match(prediction, reference):
    return normalize(reference) in normalize(prediction)

def compute_rouge_l(prediction, reference):
    return rouge_l_scorer.score(reference, prediction)['rougeL'].fmeasure

def compute_f1_token(prediction, reference):
    pred_tokens = normalize(extract_answer(prediction)).split()
    ref_tokens = normalize(reference).split()
    if not pred_tokens or not ref_tokens:
        return 0.0
    pred_counts, ref_counts = Counter(pred_tokens), Counter(ref_tokens)
    common = sum((pred_counts & ref_counts).values())
    if not common:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

## 4. Compute metrics

Drop `no_answer` rows, score the rest. BERTScore runs once over every row for speed.

In [ ]:
scored = df[df["answer_type"] != "no_answer"].copy()

scored["extracted_answer"] = scored["response"].apply(extract_answer)
scored["exact_match"]    = scored.apply(lambda r: exact_match(r["response"], r["reference_answer"]), axis=1)
scored["contains_match"] = scored.apply(lambda r: contains_match(r["response"], r["reference_answer"]), axis=1)
scored["rouge_l"]        = scored.apply(lambda r: compute_rouge_l(r["response"], r["reference_answer"]), axis=1)
scored["f1_token"]       = scored.apply(lambda r: compute_f1_token(r["response"], r["reference_answer"]), axis=1)
scored["has_marker"]     = scored["response"].str.contains(ANSWER_MARKER, regex=False)

print(f"Scored rows: {len(scored)}  ({scored['model'].nunique()} models x 2 techniques x 2 sources)")

In [ ]:
preds = scored["response"].tolist()
refs  = scored["reference_answer"].tolist()
print(f"Computing BERTScore for {len(preds)} pairs...")
_, _, F1 = compute_bert_score(preds, refs, lang="en", verbose=True)
scored["bertscore_f1"] = F1.numpy()
print("Done.")

## 5. Results — fixed vs. dynamic tables

In [ ]:
overall = (scored.groupby("source", observed=True)[METRICS]
           .mean().reindex(SOURCE_ORDER).round(4).rename(columns=METRIC_LABELS))
print("Headline: mean over all models and both techniques")
overall

In [ ]:
by_base = (scored.groupby(["base", "source"], observed=True)[METRICS]
           .mean().round(4).rename(columns=METRIC_LABELS))
print("By technique and source:")
by_base

In [ ]:
by_all = (scored.groupby(["model", "base", "source"], observed=True)[METRICS]
          .mean().round(4).rename(columns=METRIC_LABELS))
print("Full breakdown by model, technique and source:")
by_all

In [ ]:
# The headline number the comparison is about: dynamic minus fixed.
by_mbs = scored.groupby(["model", "base", "source"], observed=True)[METRICS].mean()
delta = (by_mbs.xs("dynamic", level="source") - by_mbs.xs("fixed", level="source")).round(4)
delta = delta.rename(columns=METRIC_LABELS)
print("Dynamic − fixed  (positive = dynamic wins), by model and technique:")
delta

## 6. Graphs

In [ ]:
plot_df = scored.groupby("source", observed=True)[METRICS].mean().reindex(SOURCE_ORDER).T
plot_df.index = [METRIC_LABELS[m] for m in METRICS]
ax = plot_df.plot(kind="bar", figsize=(13, 6), width=0.8,
                  color={"fixed": "#4C72B0", "dynamic": "#DD8452"})
ax.set_title("Fixed vs. dynamic — overall metric scores")
ax.set_xlabel(""); ax.set_ylabel("Score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.legend(title="Source")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
for ax, metric in zip(axes.flat, METRICS):
    pivot = (scored.groupby(["model", "source"], observed=True)[metric]
             .mean().unstack("source").reindex(MODEL_ORDER)[SOURCE_ORDER])
    pivot.plot(kind="bar", ax=ax, width=0.8,
               color={"fixed": "#4C72B0", "dynamic": "#DD8452"}, legend=False)
    ax.set_title(METRIC_LABELS[metric])
    ax.set_xlabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
axes.flat[-1].axis("off")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Source", loc="lower right", bbox_to_anchor=(0.97, 0.08))
fig.suptitle("Each metric: fixed vs. dynamic, by model", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Improvement heatmap: dynamic − fixed, one panel per technique.
fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
for ax, base in zip(axes, BASE_ORDER):
    sub = scored[scored["base"] == base]
    means = sub.groupby(["model", "source"], observed=True)[METRICS].mean()
    d = (means.xs("dynamic", level="source") - means.xs("fixed", level="source")).reindex(MODEL_ORDER)
    d.columns = [METRIC_LABELS[m] for m in METRICS]
    sns.heatmap(d, annot=True, fmt="+.3f", cmap="RdBu", center=0, ax=ax, cbar=True)
    ax.set_title(f"{BASE_LABELS[base]}: dynamic − fixed")
    ax.set_xlabel(""); ax.set_ylabel("")
fig.suptitle("Where dynamic helps (blue) or hurts (red)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Does dynamic help more on some answer types?
pivot = (scored.groupby(["answer_type", "source"], observed=True)["contains_match"]
         .mean().unstack("source")[SOURCE_ORDER].reindex(ANSWER_TYPE_ORDER))
ax = pivot.plot(kind="bar", figsize=(11, 5.5), width=0.8,
                color={"fixed": "#4C72B0", "dynamic": "#DD8452"})
ax.set_title("Contains Match by answer type — fixed vs. dynamic")
ax.set_xlabel(""); ax.set_ylabel("Contains Match")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Source")
plt.tight_layout()
plt.show()

## 7. Cost of dynamic prompts

Dynamic prompts are longer (3 retrieved examples with reasoning), so they cost more to run. This is the overhead side of any accuracy gain.

In [ ]:
cost = (df.groupby("source", observed=True)
        .agg(avg_gen_time=("generation_time", "mean"),
             avg_resp_len=("response_length", "mean"))
        .reindex(SOURCE_ORDER).round(2))
print(cost)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cost["avg_gen_time"].plot(kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Avg generation time (s)"); axes[0].set_xlabel("")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
cost["avg_resp_len"].plot(kind="bar", ax=axes[1], color=["#4C72B0", "#DD8452"])
axes[1].set_title("Avg response length (tokens)"); axes[1].set_xlabel("")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 8. Qualitative comparison

Same question, fixed vs. dynamic side by side for one model.

In [ ]:
def show_comparison(sample_id, model_label, base="few_shot", max_chars=600):
    rows = scored[(scored["sample_id"] == sample_id)
                  & (scored["model"] == model_label)
                  & (scored["base"] == base)]
    if rows.empty:
        print(f"No scored rows for sample_id={sample_id}, {model_label}, {base}")
        return
    first = rows.iloc[0]
    print(f"QUESTION (sample_id={sample_id}, {model_label}, {BASE_LABELS[base]}):\n{first['question']}\n")
    print(f"GOLD REFERENCE:\n{first['reference_answer']}\n")
    print("=" * 90)
    for source in SOURCE_ORDER:
        r = rows[rows["source"] == source]
        if r.empty:
            continue
        r = r.iloc[0]
        resp = r["response"][:max_chars] + ("..." if len(r["response"]) > max_chars else "")
        print(f"\n[{source}]  contains={r['contains_match']}  bertscore_f1={r['bertscore_f1']:.3f}")
        print(resp)
        print("-" * 90)

model_label = MODEL_ORDER[0]
example_id = scored[scored["answer_type"] == "short"]["sample_id"].iloc[0]
show_comparison(example_id, model_label, base="few_shot")